# MalthusJAX Level 2 Demo: Genetic Operators

This notebook demonstrates **Level 2 genetic operators** that work seamlessly with all genome types from Level 1, showcasing the flexibility and power of MalthusJAX's abstract architecture.

## Overview

Level 2 provides a comprehensive suite of evolutionary operators:
- **Mutation operators**: Bit flip, Gaussian, categorical mutations
- **Crossover operators**: Uniform, single-point, blend, SBX
- **Selection operators**: Tournament, roulette wheel

## Architecture Design

### Key Features

1. **Unified Factory Pattern**: All operators use `@struct.dataclass` with `__call__` methods
2. **Static/Dynamic Separation**: Compilation parameters vs. runtime tunable parameters
3. **Automatic Vectorization**: Built-in support for generating multiple offspring
4. **Batch-First Output**: Operators return `(num_offspring, genome_shape)` for seamless pipeline integration
5. **Generic Type Support**: Operators work across binary, real, and categorical genomes

### Design Pattern

```python
operator = MutationOperator(num_offspring=3, static_param=value)
offspring_batch = operator(key, genome, config)  # Returns (3, ...genome_shape)
```

This architecture eliminates glue code and enables clean operator composition in evolution engines.

In [1]:
# Essential imports
import jax
import jax.numpy as jnp
import jax.random as jar
from flax import struct
import time

# Core genome types
from malthusjax.core.genome.binary_genome import BinaryGenome, BinaryGenomeConfig
from malthusjax.core.genome.real_genome import RealGenome, RealGenomeConfig  
from malthusjax.core.genome.categorical_genome import CategoricalGenome, CategoricalGenomeConfig

# Mutation operators
from malthusjax.operators.mutation.binary import BitFlipMutation, ScrambleMutation as BinaryScramble
from malthusjax.operators.mutation.real import GaussianMutation, BallMutation, PolynomialMutation
from malthusjax.operators.mutation.categorical import CategoricalFlipMutation, RandomCategoryMutation

# Crossover operators
from malthusjax.operators.crossover.binary import UniformCrossover, SinglePointCrossover
from malthusjax.operators.crossover.real import BlendCrossover, SimulatedBinaryCrossover

# Selection operators
from malthusjax.operators.selection.tournament import TournamentSelection
from malthusjax.operators.selection.roulette import RouletteWheelSelection

# Base classes
from malthusjax.operators.base import BaseMutation, BaseCrossover, BaseSelection

print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print("Level 2 operators loaded successfully")

# Initialize random key
key = jar.PRNGKey(42)

JAX version: 0.8.0
JAX backend: cpu
Level 2 operators loaded successfully


## Abstract Base Classes

The Level 2 architecture is built on three abstract base classes that define consistent interfaces for all genetic operators.

In [2]:
print("Abstract Base Classes:")
print(f"  BaseMutation: {BaseMutation.__doc__.split('.')[0] if BaseMutation.__doc__ else 'Mutation operator interface'}")  
print(f"  BaseCrossover: {BaseCrossover.__doc__.split('.')[0] if BaseCrossover.__doc__ else 'Crossover operator interface'}")
print(f"  BaseSelection: {BaseSelection.__doc__.split('.')[0] if BaseSelection.__doc__ else 'Selection operator interface'}")

print("\nDesign pattern:")
print("  - Static parameters (num_offspring, tournament_size) set at creation")
print("  - Dynamic parameters (mutation_rate, crossover_rate) passed to methods")  
print("  - Factory __call__ methods delegate to pure JAX functions")
print("  - Automatic vectorization via jax.vmap() for batch operations")

Abstract Base Classes:
  BaseMutation: 
    Abstract Mutation Operator using the new paradigm
  BaseCrossover: 
    Abstract Crossover Operator using the new paradigm
  BaseSelection: 
    Abstract Selection Operator using the new paradigm

Design pattern:
  - Static parameters (num_offspring, tournament_size) set at creation
  - Dynamic parameters (mutation_rate, crossover_rate) passed to methods
  - Factory __call__ methods delegate to pure JAX functions
  - Automatic vectorization via jax.vmap() for batch operations


## Part 1: Binary Genome Mutations

Demonstrating mutation operators on binary genomes with automatic offspring generation.

In [3]:
# Create binary genome
key1, key2 = jar.split(key)
binary_config = BinaryGenomeConfig(length=10)
binary_genome = BinaryGenome.random_init(key1, binary_config)

print(f"Original genome: {binary_genome}")

# BitFlip mutation with multiple offspring
bitflip_mutator = BitFlipMutation(num_offspring=3, mutation_rate=0.3)
mutated_offspring = bitflip_mutator(key2, binary_genome, binary_config)

print(f"\nBitFlip mutation results (3 offspring, rate=0.3):")
print(f"Batch shape: {mutated_offspring.bits.shape}")
for i in range(mutated_offspring.bits.shape[0]):
    child = BinaryGenome(bits=mutated_offspring.bits[i])
    print(f"  Offspring {i+1}: {child}")

# Scramble mutation (permutation-based)
key3 = jar.split(key2)[0]
scramble_mutator = BinaryScramble(num_offspring=2)
scrambled_offspring = scramble_mutator(key3, binary_genome, binary_config)

print(f"\nScramble mutation results (2 offspring):")
print(f"Batch shape: {scrambled_offspring.bits.shape}")
for i in range(scrambled_offspring.bits.shape[0]):
    child = BinaryGenome(bits=scrambled_offspring.bits[i])
    print(f"  Scrambled {i+1}: {child}")

print(f"\nBatch-first output enables direct pipeline integration with crossover/selection operators")

Original genome: <BinaryGenome(0100000110, len=10)>

BitFlip mutation results (3 offspring, rate=0.3):
Batch shape: (3, 10)
  Offspring 1: <BinaryGenome(1101100110, len=10)>
  Offspring 2: <BinaryGenome(1110100110, len=10)>
  Offspring 3: <BinaryGenome(0101010110, len=10)>

BitFlip mutation results (3 offspring, rate=0.3):
Batch shape: (3, 10)
  Offspring 1: <BinaryGenome(1101100110, len=10)>
  Offspring 2: <BinaryGenome(1110100110, len=10)>
  Offspring 3: <BinaryGenome(0101010110, len=10)>

Scramble mutation results (2 offspring):
Batch shape: (2, 10)
  Scrambled 1: <BinaryGenome(0100000110, len=10)>
  Scrambled 2: <BinaryGenome(0100100010, len=10)>

Batch-first output enables direct pipeline integration with crossover/selection operators

Scramble mutation results (2 offspring):
Batch shape: (2, 10)
  Scrambled 1: <BinaryGenome(0100000110, len=10)>
  Scrambled 2: <BinaryGenome(0100100010, len=10)>

Batch-first output enables direct pipeline integration with crossover/selection operat

## Part 2: Real Genome Mutations

Real-valued mutations with boundary constraint enforcement.

In [4]:
# Create real genome
key4, key5 = jar.split(key3)
real_config = RealGenomeConfig(length=5, bounds=(-5.0, 5.0))
real_genome = RealGenome.random_init(key4, real_config)

print(f"Original genome: {real_genome}")
print(f"Bounds: {real_config.bounds}")

# Gaussian mutation
gaussian_mutator = GaussianMutation(num_offspring=4, mutation_rate=0.4, mutation_strength=0.5)
gaussian_offspring = gaussian_mutator(key5, real_genome, real_config)

print(f"\nGaussian mutation results (4 offspring, σ=0.5):")
print(f"Batch shape: {gaussian_offspring.values.shape}")
for i in range(gaussian_offspring.values.shape[0]):
    child = RealGenome(values=gaussian_offspring.values[i])
    print(f"  Offspring {i+1}: {child}")

# Ball mutation (uniform within hypersphere)
key6 = jar.split(key5)[0]  
ball_mutator = BallMutation(num_offspring=2, mutation_strength=1.0)
ball_offspring = ball_mutator(key6, real_genome, real_config)

print(f"\nBall mutation results (2 offspring, strength=1.0):")
print(f"Batch shape: {ball_offspring.values.shape}")
for i in range(ball_offspring.values.shape[0]):
    child = RealGenome(values=ball_offspring.values[i])
    print(f"  Offspring {i+1}: {child}")

# Polynomial mutation
key7 = jar.split(key6)[0]
poly_mutator = PolynomialMutation(num_offspring=3, mutation_rate=0.6, eta=20.0)
poly_offspring = poly_mutator(key7, real_genome, real_config)

print(f"\nPolynomial mutation results (3 offspring, η=20):")
print(f"Batch shape: {poly_offspring.values.shape}")
for i in range(poly_offspring.values.shape[0]):
    child = RealGenome(values=poly_offspring.values[i])
    print(f"  Offspring {i+1}: {child}")

print(f"\nAll mutations automatically enforce boundary constraints")

Original genome: <RealGenome([-4.341, -0.092, -3.379, 2.417, 1.235], len=5)>
Bounds: (-5.0, 5.0)

Gaussian mutation results (4 offspring, σ=0.5):
Batch shape: (4, 5)
  Offspring 1: <RealGenome([-4.941, -0.092, -3.379, 2.417, 1.235], len=5)>
  Offspring 2: <RealGenome([-4.341, -0.092, -3.379, 2.417, 1.235], len=5)>
  Offspring 3: <RealGenome([-4.341, -0.092, -3.379, 2.565, 1.339], len=5)>
  Offspring 4: <RealGenome([-4.174, -0.238, -3.379, 2.341, 1.235], len=5)>

Ball mutation results (2 offspring, strength=1.0):
Batch shape: (2, 5)

Gaussian mutation results (4 offspring, σ=0.5):
Batch shape: (4, 5)
  Offspring 1: <RealGenome([-4.941, -0.092, -3.379, 2.417, 1.235], len=5)>
  Offspring 2: <RealGenome([-4.341, -0.092, -3.379, 2.417, 1.235], len=5)>
  Offspring 3: <RealGenome([-4.341, -0.092, -3.379, 2.565, 1.339], len=5)>
  Offspring 4: <RealGenome([-4.174, -0.238, -3.379, 2.341, 1.235], len=5)>

Ball mutation results (2 offspring, strength=1.0):
Batch shape: (2, 5)
  Offspring 1: <RealG

## Part 3: Categorical Genome Mutations

Discrete mutations for architecture search and combinatorial optimization.

In [5]:
# Create categorical genome
key8, key9 = jar.split(key7)
categorical_config = CategoricalGenomeConfig(length=6, num_categories=8)
categorical_genome = CategoricalGenome.random_init(key8, categorical_config)

print(f"Original genome: {categorical_genome}")
print(f"Categories per position: {categorical_config.num_categories}")

# Categorical flip mutation
categorical_flip = CategoricalFlipMutation(num_offspring=3, mutation_rate=0.5)
flip_offspring = categorical_flip(key9, categorical_genome, categorical_config)

print(f"\nCategorical flip mutation results (3 offspring, rate=0.5):")
print(f"Batch shape: {flip_offspring.categories.shape}")
for i in range(flip_offspring.categories.shape[0]):
    child = CategoricalGenome(categories=flip_offspring.categories[i])
    print(f"  Offspring {i+1}: {child}")

# Random category mutation
key10 = jar.split(key9)[0]
random_cat = RandomCategoryMutation(num_offspring=2, mutation_rate=0.3)
random_offspring = random_cat(key10, categorical_genome, categorical_config)

print(f"\nRandom category mutation results (2 offspring, rate=0.3):")
print(f"Batch shape: {random_offspring.categories.shape}")
for i in range(random_offspring.categories.shape[0]):
    child = CategoricalGenome(categories=random_offspring.categories[i])
    print(f"  Offspring {i+1}: {child}")

print(f"\nCategory constraints automatically enforced (0 to {categorical_config.num_categories-1})")

Original genome: <CategoricalGenome([2, 3, 3, 6, 6, 1], len=6)>
Categories per position: 8

Categorical flip mutation results (3 offspring, rate=0.5):
Batch shape: (3, 6)
  Offspring 1: <CategoricalGenome([2, 5, 3, 0, 6, 1], len=6)>
  Offspring 2: <CategoricalGenome([7, 3, 3, 5, 2, 3], len=6)>
  Offspring 3: <CategoricalGenome([2, 3, 2, 6, 6, 1], len=6)>

Categorical flip mutation results (3 offspring, rate=0.5):
Batch shape: (3, 6)
  Offspring 1: <CategoricalGenome([2, 5, 3, 0, 6, 1], len=6)>
  Offspring 2: <CategoricalGenome([7, 3, 3, 5, 2, 3], len=6)>
  Offspring 3: <CategoricalGenome([2, 3, 2, 6, 6, 1], len=6)>

Random category mutation results (2 offspring, rate=0.3):
Batch shape: (2, 6)
  Offspring 1: <CategoricalGenome([2, 3, 4, 6, 6, 1], len=6)>
  Offspring 2: <CategoricalGenome([2, 3, 3, 6, 6, 1], len=6)>

Category constraints automatically enforced (0 to 7)

Random category mutation results (2 offspring, rate=0.3):
Batch shape: (2, 6)
  Offspring 1: <CategoricalGenome([2, 3, 

## Part 4: JIT Compilation Performance

Demonstrating the performance benefits of JAX JIT compilation with genetic operators.

In [6]:
# Performance testing setup
key_perf = jar.split(key10)[0]
large_binary_config = BinaryGenomeConfig(length=1000)
large_genome = BinaryGenome.random_init(key_perf, large_binary_config)

perf_mutator = BitFlipMutation(num_offspring=100, mutation_rate=0.1)

print("Performance benchmark:")
print(f"  Genome length: {large_binary_config.length}")
print(f"  Offspring count: {perf_mutator.num_offspring}")
print(f"  Mutation rate: {perf_mutator.mutation_rate}")

# First run (includes compilation)
start_time = time.time()
offspring_1 = perf_mutator(key_perf, large_genome, large_binary_config)
compile_time = time.time() - start_time

# Second run (uses cached compilation)
key_perf2 = jar.split(key_perf)[0] 
start_time = time.time()
offspring_2 = perf_mutator(key_perf2, large_genome, large_binary_config)
jit_time = time.time() - start_time

print(f"\nResults:")
print(f"  First run (with compilation): {compile_time:.4f}s")
print(f"  Cached JIT run: {jit_time:.4f}s")
print(f"  Speedup factor: {compile_time/jit_time:.1f}x")
print(f"  Output shape: {offspring_1.bits.shape}")

print(f"\nVectorization validated:")
print(f"  Batch size: {offspring_1.bits.shape[0]} offspring")
print(f"  Genome length: {offspring_1.bits.shape[1]} genes")
print(f"  Single vectorized operation replaces {offspring_1.bits.shape[0]} sequential calls")

Performance benchmark:
  Genome length: 1000
  Offspring count: 100
  Mutation rate: 0.1

Results:
  First run (with compilation): 0.1743s
  Cached JIT run: 0.0007s
  Speedup factor: 256.7x
  Output shape: (100, 1000)

Vectorization validated:
  Batch size: 100 offspring
  Genome length: 1000 genes
  Single vectorized operation replaces 100 sequential calls

Results:
  First run (with compilation): 0.1743s
  Cached JIT run: 0.0007s
  Speedup factor: 256.7x
  Output shape: (100, 1000)

Vectorization validated:
  Batch size: 100 offspring
  Genome length: 1000 genes
  Single vectorized operation replaces 100 sequential calls


## Part 5: Crossover Operators

Batch-first crossover operators for recombination.

In [7]:
# Binary crossover demonstration
key_fresh = jar.PRNGKey(999)
key_cross1, key_cross2, key_cross3 = jar.split(key_fresh, 3)

parent1 = BinaryGenome.random_init(key_cross1, binary_config)
parent2 = BinaryGenome.random_init(key_cross2, binary_config)

print("Binary crossover:")
print(f"Parent 1: {parent1}")
print(f"Parent 2: {parent2}")

# Uniform crossover with batch output
uniform_cross = UniformCrossover(num_offspring=3, crossover_rate=0.7)
uniform_offspring = uniform_cross(key_cross3, parent1, parent2, binary_config)

print(f"\nUniform crossover results (3 offspring, rate=0.7):")
print(f"Batch shape: {uniform_offspring.bits.shape}")
for i in range(uniform_offspring.bits.shape[0]):
    child = BinaryGenome(bits=uniform_offspring.bits[i])
    print(f"  Offspring {i+1}: {child}")

# Single point crossover
key_cross4 = jar.PRNGKey(555)
single_point = SinglePointCrossover(num_offspring=2)
single_offspring = single_point(key_cross4, parent1, parent2, binary_config)

print(f"\nSingle point crossover results (2 offspring):")
print(f"Batch shape: {single_offspring.bits.shape}")
for i in range(single_offspring.bits.shape[0]):
    child = BinaryGenome(bits=single_offspring.bits[i])
    print(f"  Offspring {i+1}: {child}")

# Real genome crossover
key_real1, key_real2, key_real3 = jar.split(jar.PRNGKey(333), 3)
real_parent1 = RealGenome.random_init(key_real1, real_config)
real_parent2 = RealGenome.random_init(key_real2, real_config)

print(f"\nReal genome crossover:")
print(f"Parent 1: {real_parent1}")
print(f"Parent 2: {real_parent2}")

# Blend crossover
blend_cross = BlendCrossover(num_offspring=2, crossover_rate=0.8, alpha=0.3)
blend_offspring = blend_cross(key_real3, real_parent1, real_parent2, real_config)

print(f"\nBlend crossover results (2 offspring, α=0.3):")
print(f"Batch shape: {blend_offspring.values.shape}")
for i in range(blend_offspring.values.shape[0]):
    child = RealGenome(values=blend_offspring.values[i])
    print(f"  Offspring {i+1}: {child}")

# Simulated Binary Crossover
key_cross5 = jar.PRNGKey(111)
sbx_cross = SimulatedBinaryCrossover(num_offspring=1, eta=15.0)
sbx_offspring = sbx_cross(key_cross5, real_parent1, real_parent2, real_config)

print(f"\nSBX crossover (η=15.0, 1 offspring):")
print(f"Batch shape: {sbx_offspring.values.shape}")
child = RealGenome(values=sbx_offspring.values[0])
print(f"  Offspring: {child}")

print(f"\nBatch-first paradigm eliminates tuple unpacking:")
print(f"  Output: (num_offspring, genome_shape)")
print(f"  Direct pipeline: crossover → mutation → evaluation")

Binary crossover:
Parent 1: <BinaryGenome(1101101001, len=10)>
Parent 2: <BinaryGenome(0111111011, len=10)>

Uniform crossover results (3 offspring, rate=0.7):
Batch shape: (3, 10)
  Offspring 1: <BinaryGenome(1101111001, len=10)>
  Offspring 2: <BinaryGenome(1111101001, len=10)>
  Offspring 3: <BinaryGenome(1101101001, len=10)>

Single point crossover results (2 offspring):
Batch shape: (2, 10)
  Offspring 1: <BinaryGenome(1101101001, len=10)>
  Offspring 2: <BinaryGenome(1101111011, len=10)>

Real genome crossover:
Parent 1: <RealGenome([-2.690, 4.494, 3.688, -1.386, -1.962], len=5)>
Parent 2: <RealGenome([-2.510, 3.432, -3.442, 0.158, 4.229], len=5)>

Single point crossover results (2 offspring):
Batch shape: (2, 10)
  Offspring 1: <BinaryGenome(1101101001, len=10)>
  Offspring 2: <BinaryGenome(1101111011, len=10)>

Real genome crossover:
Parent 1: <RealGenome([-2.690, 4.494, 3.688, -1.386, -1.962], len=5)>
Parent 2: <RealGenome([-2.510, 3.432, -3.442, 0.158, 4.229], len=5)>

Blend 

## Part 6: Selection Operators

Genome-agnostic selection based on fitness values.

In [8]:
# Create fitness values for selection
population_size = 20
key_select = jar.split(key10)[0]
fitness_values = jar.uniform(key_select, (population_size,), minval=0.0, maxval=100.0)

print("Selection demonstration:")
print(f"Population size: {population_size}")
print(f"Fitness values: {jnp.round(fitness_values, 1)}")
print(f"Best: {jnp.max(fitness_values):.1f} (index {jnp.argmax(fitness_values)})")
print(f"Worst: {jnp.min(fitness_values):.1f} (index {jnp.argmin(fitness_values)})")

# Tournament selection
tournament_selector = TournamentSelection(num_selections=5, tournament_size=4)
key_tour = jar.split(key_select)[0]
selected_indices = tournament_selector(key_tour, fitness_values)

print(f"\nTournament selection (5 selections, tournament size 4):")
print(f"Selected indices: {selected_indices}")
print(f"Selected fitness: {jnp.round(fitness_values[selected_indices], 1)}")

# Roulette wheel selection
roulette_selector = RouletteWheelSelection(num_selections=6)
key_roul = jar.split(key_tour)[0]
roulette_indices = roulette_selector(key_roul, fitness_values)

print(f"\nRoulette wheel selection (6 selections):")
print(f"Selected indices: {roulette_indices}")
print(f"Selected fitness: {jnp.round(fitness_values[roulette_indices], 1)}")

# Selection pressure analysis
tournament_avg = jnp.mean(fitness_values[selected_indices])
roulette_avg = jnp.mean(fitness_values[roulette_indices])
population_avg = jnp.mean(fitness_values)

print(f"\nSelection pressure comparison:")
print(f"  Population average: {population_avg:.1f}")
print(f"  Tournament average: {tournament_avg:.1f} (pressure: {tournament_avg/population_avg:.2f}x)")
print(f"  Roulette average: {roulette_avg:.1f} (pressure: {roulette_avg/population_avg:.2f}x)")

# JIT compilation verification
jit_tournament = jax.jit(tournament_selector)
jit_roulette = jax.jit(roulette_selector)

jit_tournament_result = jit_tournament(jar.PRNGKey(999), fitness_values)
jit_roulette_result = jit_roulette(jar.PRNGKey(888), fitness_values)

print(f"\nJIT compilation verified:")
print(f"  Tournament JIT result: {jit_tournament_result}")
print(f"  Roulette JIT result: {jit_roulette_result}")

Selection demonstration:
Population size: 20
Fitness values: [94.8        49.9        39.8         0.90000004 99.200005   51.4
 79.4        31.         35.7        61.2         7.3        21.300001
 47.100002   52.600002   29.300001   87.5        54.100002   94.700005
 27.5        96.9       ]
Best: 99.2 (index 4)
Worst: 0.9 (index 3)

Tournament selection (5 selections, tournament size 4):
Selected indices: [15 19  4  4 19]
Selected fitness: [87.5      96.9      99.200005 99.200005 96.9     ]

Tournament selection (5 selections, tournament size 4):
Selected indices: [15 19  4  4 19]
Selected fitness: [87.5      96.9      99.200005 99.200005 96.9     ]

Roulette wheel selection (6 selections):
Selected indices: [ 1  9  7 16  0 13]
Selected fitness: [49.9      61.2      31.       54.100002 94.8      52.600002]

Selection pressure comparison:
  Population average: 53.1
  Tournament average: 96.0 (pressure: 1.81x)
  Roulette average: 57.3 (pressure: 1.08x)

Roulette wheel selection (6 sel

## Part 7: Complete Evolution Pipeline

Demonstrating operator composition: selection → crossover → mutation.

## 🔬 Complete Evolutionary Generation Demo

Let's put it all together! A complete evolutionary generation using our Level 2 operators: **Selection** → **Crossover** → **Mutation** with the new paradigm.

In [9]:
# Complete generation demo
pop_size = 8
key_evo = jar.split(key_roul)[0]

# Initialize population
pop_keys = jar.split(key_evo, pop_size)
population = [BinaryGenome.random_init(k, binary_config) for k in pop_keys]

print("Complete evolutionary generation pipeline")
print(f"Population size: {pop_size}")
print("\nInitial population:")
for i, genome in enumerate(population):
    print(f"  Individual {i}: {genome}")

# Step 1: Fitness evaluation
fitness_values = jnp.array([jnp.sum(genome.bits) for genome in population])
print(f"\nStep 1 - Fitness evaluation:")
print(f"  Fitness values: {fitness_values}")
print(f"  Best: {jnp.max(fitness_values)} (Individual {jnp.argmax(fitness_values)})")

# Step 2: Selection
key_sel = jar.split(key_evo)[1]
selector = TournamentSelection(num_selections=4, tournament_size=3)
selected_indices = selector(key_sel, fitness_values)

print(f"\nStep 2 - Tournament selection:")
print(f"  Selected indices: {selected_indices}")
print(f"  Selected fitness: {fitness_values[selected_indices]}")

# Step 3: Crossover (batch-first output)
key_cross = jar.split(key_sel)[0]
crossover_op = UniformCrossover(num_offspring=4, crossover_rate=0.7)

parent1 = population[selected_indices[0]]
parent2 = population[selected_indices[1]]
offspring_batch = crossover_op(key_cross, parent1, parent2, binary_config)

print(f"\nStep 3 - Uniform crossover:")
print(f"  Parents: Individual {selected_indices[0]} + Individual {selected_indices[1]}")
print(f"  Offspring batch shape: {offspring_batch.bits.shape}")
print("  Crossover offspring:")
for i in range(offspring_batch.bits.shape[0]):
    child = BinaryGenome(bits=offspring_batch.bits[i])
    print(f"    Offspring {i}: {child}")

# Step 4: Mutation (batch-compatible)
key_mut = jar.split(key_cross)[0]
mutator = BitFlipMutation(num_offspring=1, mutation_rate=0.2)

final_offspring = []
for i in range(offspring_batch.bits.shape[0]):
    child_genome = BinaryGenome(bits=offspring_batch.bits[i])
    key_mut, subkey = jar.split(key_mut)
    mutated = mutator(subkey, child_genome, binary_config)
    final_offspring.append(mutated.bits[0])

print(f"\nStep 4 - Bit flip mutation:")
print("  Final offspring after mutation:")
for i, bits in enumerate(final_offspring):
    fitness = jnp.sum(bits)
    print(f"    Offspring {i}: fitness={fitness}")

# Generation summary
original_avg = jnp.mean(fitness_values)
offspring_fitness = [jnp.sum(bits) for bits in final_offspring]
offspring_avg = jnp.mean(jnp.array(offspring_fitness))

print(f"\nGeneration summary:")
print(f"  Original average fitness: {original_avg:.1f}")
print(f"  Offspring average fitness: {offspring_avg:.1f}")
print(f"  Improvement: {offspring_avg - original_avg:+.1f}")
print(f"  Best original: {jnp.max(fitness_values)}")
print(f"  Best offspring: {jnp.max(jnp.array(offspring_fitness))}")

Complete evolutionary generation pipeline
Population size: 8

Initial population:
  Individual 0: <BinaryGenome(1111100110, len=10)>
  Individual 1: <BinaryGenome(1111001000, len=10)>
  Individual 2: <BinaryGenome(1101111001, len=10)>
  Individual 3: <BinaryGenome(1110011010, len=10)>
  Individual 4: <BinaryGenome(1100101000, len=10)>
  Individual 5: <BinaryGenome(0110010010, len=10)>
  Individual 6: <BinaryGenome(0100100001, len=10)>
  Individual 7: <BinaryGenome(1110010110, len=10)>

Step 1 - Fitness evaluation:
  Fitness values: [7 5 7 6 4 4 3 6]
  Best: 7 (Individual 0)

Step 2 - Tournament selection:
  Selected indices: [2 2 2 1]
  Selected fitness: [7 7 7 5]

Step 3 - Uniform crossover:
  Parents: Individual 2 + Individual 2
  Offspring batch shape: (4, 10)
  Crossover offspring:

Step 2 - Tournament selection:
  Selected indices: [2 2 2 1]
  Selected fitness: [7 7 7 5]

Step 3 - Uniform crossover:
  Parents: Individual 2 + Individual 2
  Offspring batch shape: (4, 10)
  Crossove

## Summary: Level 2 Architecture Benefits

### Unified Operator Interface

All genetic operators follow a consistent design:
- **Immutable configuration**: `@struct.dataclass` for JAX compatibility
- **Factory pattern**: `__call__` methods for clean invocation
- **Batch-first output**: Direct pipeline integration without glue code
- **Type generic**: Work across all genome types from Level 1

### Static vs. Dynamic Parameters

**Static parameters** (set at creation, control JIT compilation):
- `num_offspring`: Number of offspring to generate
- `tournament_size`: Tournament selection pressure
- `eta`: Distribution index for SBX

**Dynamic parameters** (runtime tunable without recompilation):
- `mutation_rate`: Probability of mutation per gene
- `crossover_rate`: Probability of crossover application
- `mutation_strength`: Magnitude of perturbation

### Architecture Flow

```python
# Before: Tuple unpacking and manual batching
offspring_tuple = crossover(key, p1, p2)
offspring_batch = jnp.stack(offspring_tuple)
mutated = mutation(key, offspring_batch)

# After: Direct batch flow
offspring = crossover(key, p1, p2, config)  # Returns (n_offspring, genome_shape)
mutated = mutation(key, offspring, config)  # Accepts batch directly
```

### Performance Characteristics

- **JIT compilation**: 6x speedup after initial compilation
- **Vectorization**: Single operation replaces sequential loops
- **Memory efficiency**: Structured PyTree outputs
- **GPU acceleration**: Automatic when JAX GPU backend is available

### Extensibility

New operators can be added by:
1. Inheriting from `BaseMutation`, `BaseCrossover`, or `BaseSelection`
2. Implementing the `__call__` method following the signature pattern
3. Using `@struct.dataclass` for immutability
4. Returning batch-first output: `(num_offspring, ...genome_shape)`

The consistent interface enables seamless integration with Level 3 evolution engines.